# Phase 5 — TF-IDF & Cosine Similarity

## Information Retrieval — Ranked Search

This phase focuses on ranking programming questions according to their relevance to a user's query using TF-IDF weighting and Cosine Similarity.

### Objectives
- Understand TF-IDF weighting.
- Represent documents and queries numerically.
- Calculate Cosine Similarity.
- Rank retrieved documents by relevance.
- Retrieve the Top-K most relevant questions.

In [27]:
import pandas as pd
import numpy as np

from collections import Counter
import math

import re

In [28]:
# Load the processed documents from Phase 2

data_path = "../Data/artifacts/phase2/processed_documents.csv.gz"

df = pd.read_csv(data_path)

df.head()

,Id,Title,Body,Tags,CreationDate,Y,Document,LowercasedDocument,RawTokens,FilteredTokens,Tokens,ProcessedDocument,TokenCount
0,34552656,Java: Repeat Task Every Random Seconds,I'm already familiar with repeating tasks ever...,java repeat,2016-01-01 00:21:59,LQ_CLOSE,Java: Repeat Task Every Random Seconds I'm alr...,java: repeat task every random seconds i'm alr...,"['java', 'repeat', 'task', 'every', 'random', ...","['java', 'repeat', 'task', 'every', 'random', ...","[""java"", ""repeat"", ""task"", ""every"", ""random"", ...",java repeat task every random second m already...,57
1,34553034,Why are Java Optionals immutable?,I'd like to understand why Java 8 Optionals we...,java optional,2016-01-01 02:03:20,HQ,Why are Java Optionals immutable? I'd like to ...,why are java optionals immutable? i'd like to ...,"['why', 'are', 'java', 'optionals', 'immutable...","['why', 'java', 'optionals', 'immutable', 'd',...","[""why"", ""java"", ""optionals"", ""immutable"", ""d"",...",why java optionals immutable d like to underst...,19
2,34553174,Text Overlay Image with Darkened Opacity React...,I am attempting to overlay a title over an ima...,javascript image overlay react-native opacity,2016-01-01 02:48:24,HQ,Text Overlay Image with Darkened Opacity React...,text overlay image with darkened opacity react...,"['text', 'overlay', 'image', 'with', 'darkened...","['text', 'overlay', 'image', 'with', 'darkened...","[""text"", ""overlay"", ""image"", ""with"", ""darkened...",text overlay image with darkened opacity react...,363
3,34553318,Why ternary operator in swift is so picky?,"The question is very simple, but I just could ...",swift operators whitespace ternary-operator op...,2016-01-01 03:30:17,HQ,Why ternary operator in swift is so picky? The...,why ternary operator in swift is so picky? the...,"['why', 'ternary', 'operator', 'in', 'swift', ...","['why', 'ternary', 'operator', 'in', 'swift', ...","[""why"", ""ternary"", ""operator"", ""in"", ""swift"", ...",why ternary operator in swift so picky questio...,82
4,34553755,hide/show fab with scale animation,I'm using custom floatingactionmenu. I need to...,android material-design floating-action-button,2016-01-01 05:21:48,HQ,hide/show fab with scale animation I'm using c...,hide/show fab with scale animation i'm using c...,"['hide', 'show', 'fab', 'with', 'scale', 'anim...","['hide', 'show', 'fab', 'with', 'scale', 'anim...","[""hide"", ""show"", ""fab"", ""with"", ""scale"", ""anim...",hide show fab with scale animation m using cus...,33


In [29]:
N = len(df)

print("Total Documents:", N)

Total Documents: 44997


##  TF (Term Frequency) & DF (Document Frequency)

### 1 Term Frequency (TF)
### 2 Document Frequency (DF)
### 3 Inverse Document Frequency (IDF)

In [30]:

df_counts = Counter()

for document in df["ProcessedDocument"]:
    unique_words = set(document.split())
    df_counts.update(unique_words)

print("Number of unique terms:", len(df_counts))

document_tf = []

for document in df["ProcessedDocument"]:
    words = document.split()
    tf = Counter(words)
    document_tf.append(tf)


idf_scores = {}

for term, df_value in df_counts.items():
    idf_scores[term] = math.log(N / df_value)

print("Number of IDF terms:", len(idf_scores))


Number of unique terms: 273076
Number of IDF terms: 273076


### 4. TF-IDF Weighting
\[
TFIDF(t,d)=TF(t,d)\times IDF(t)
\]

In [24]:
# Calculate TF-IDF for the All documents
document_tfidf = []

for tf_dict in document_tf:
    tfidf_dict = {}

    for term, tf_value in tf_dict.items():
        idf_value = idf_scores[term]
        tfidf_value = tf_value * idf_value

        tfidf_dict[term] = tfidf_value

    document_tfidf.append(tfidf_dict)

In [26]:
print("Total TF-IDF Documents:", len(document_tfidf))

print("\nFirst Document Terms:", len(document_tfidf[0]))
print("Second Document Terms:", len(document_tfidf[1]))

Total TF-IDF Documents: 44997

First Document Terms: 44
Second Document Terms: 13


### Sparse representation of all documents using TF-IDF weights

In [38]:
# Sparse representation of all documents using TF-IDF weights

sparse_document_vectors = document_tfidf

print("Total Documents:", len(sparse_document_vectors))
print("Terms in First Document:", len(sparse_document_vectors[0]))
print("Terms in Second Document:", len(sparse_document_vectors[1]))

Total Documents: 44997
Terms in First Document: 44
Terms in Second Document: 13


### Calculate Document Norms

In [41]:
# Calculate the norm of each sparse document vector

document_norms = []

for document_vector in sparse_document_vectors:
    norm = math.sqrt(sum(weight ** 2 for weight in document_vector.values()))
    document_norms.append(norm)

print("Total Document Norms:", len(document_norms))

Total Document Norms: 44997


### Cosine Similarity Function

In [42]:
def cosine_similarity(query_vector, document_vector, document_norm):

    # Calculate dot product
    dot_product = 0

    for term, query_weight in query_vector.items():
        if term in document_vector:
            dot_product += query_weight * document_vector[term]

    # Calculate query norm
    query_norm = math.sqrt(sum(weight ** 2 for weight in query_vector.values()))

    # Avoid division by zero
    if query_norm == 0 or document_norm == 0:
        return 0

    # Calculate cosine similarity
    similarity = dot_product / (query_norm * document_norm)

    return similarity

In [43]:
query = "java repeat task every second"

In [44]:
import nltk
from nltk.stem import WordNetLemmatizer

# Ensure WordNet is available
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet", quiet=True)

try:
    nltk.data.find("corpora/omw-1.4")
except LookupError:
    nltk.download("omw-1.4", quiet=True)

lemmatizer = WordNetLemmatizer()

# Same tokenizer used in Phase 2
TOKEN_PATTERN = re.compile(
    r"(?ix)c\+\+|c\#|f\#|\.net|[a-z0-9]+(?:[._+-][a-z0-9]+)*"
)

def tokenize(text):
    if not isinstance(text, str):
        return []
    return TOKEN_PATTERN.findall(text.lower())


# Same stopwords used in Phase 2
STOPWORDS = {
    "a", "an", "the",
    "am", "is", "are", "was", "were", "be", "been", "being",
    "do", "does", "did",
    "this", "that", "these", "those",
    "i", "me", "my", "we", "our", "you", "your", "he", "she", "they", "them",
    "it", "its",
    "please", "thanks", "thank",
    "very", "just", "also"
}

PROTECTED_TERMS = {
    "not", "no", "nor", "without",
    "if", "else", "for", "while", "in", "on", "as", "from", "with",
    "class", "return", "import", "true", "false", "null", "none"
}

STOPWORDS -= PROTECTED_TERMS


def remove_stopwords(tokens):
    return [token for token in tokens if token not in STOPWORDS]


def lemmatize_token(token):
    if not token.isalpha() or token in PROTECTED_TERMS:
        return token
    return lemmatizer.lemmatize(token)


def lemmatize_tokens(tokens):
    lemmas = [lemmatize_token(token) for token in tokens]
    return [token for token in lemmas if token not in STOPWORDS]


def preprocess_query(query):
    lowercased = query.lower()
    raw_tokens = tokenize(lowercased)
    filtered_tokens = remove_stopwords(raw_tokens)
    processed_tokens = lemmatize_tokens(filtered_tokens)

    return processed_tokens

In [45]:
query = "How to repeat a Java task every second?"

query_tokens = preprocess_query(query)

print("Original Query:", query)
print("Processed Query:", query_tokens)

Original Query: How to repeat a Java task every second?
Processed Query: ['how', 'to', 'repeat', 'java', 'task', 'every', 'second']


In [46]:
# Calculate Term Frequency for the query
query_tf = Counter(query_tokens)

# Calculate TF-IDF weights for the query
query_tfidf = {}

for term, tf_value in query_tf.items():
    if term in idf_scores:
        idf_value = idf_scores[term]
        tfidf_value = tf_value * idf_value
        query_tfidf[term] = tfidf_value

query_tfidf

{'how': 0.7722680295679782,
 'to': 0.13529496904168875,
 'repeat': 5.367243569146001,
 'java': 2.146274698132662,
 'task': 3.8695356206552063,
 'every': 3.2412820118312715,
 'second': 3.123498976174888}

In [47]:
# Calculate similarity between the query and the first document
similarity = cosine_similarity(
    query_tfidf,
    sparse_document_vectors[0],
    document_norms[0]
)

print("Cosine Similarity:", similarity)

Cosine Similarity: 0.5576719492399829


In [48]:
# Calculate cosine similarity for all documents
similarity_scores = []

for i, document_vector in enumerate(sparse_document_vectors):
    similarity = cosine_similarity(
        query_tfidf,
        document_vector,
        document_norms[i]
    )

    similarity_scores.append(similarity)

# Add similarity scores to the dataframe
df["Similarity"] = similarity_scores

# Retrieve Top 10 most relevant documents
top_10 = df.sort_values(
    by="Similarity",
    ascending=False
).head(10)

top_10[["Id", "Title", "Similarity"]]

,Id,Title,Similarity
0,34552656,Java: Repeat Task Every Random Seconds,0.557672
3928,36149241,Showing progress while waiting for all Tasks i...,0.319892
44883,60342476,Python if something repeat function,0.317784
4669,36539551,How to implement Task Scheduler Manager on C# ?,0.304720
23240,45437959,Print command giving syntax error python,0.302997
32627,50624775,The project system has encountered an error Wh...,0.294414
23076,45365407,How to cycle through tasks and keep track of i...,0.292634
6082,37330574,[c#]Find dublicates in array,0.291647
3054,35755446,python coin flip program that needs a repeat l...,0.287208
21182,44324060,What does Gradle 'build' task include exactly,0.280753
